# FNB Data Centre — 3-D VTK Visualisation

Loads the `result.vtk` exported by the `export-vtk` pipeline stage and
produces interactive 3-D views of the FNB site airflow and thermal field.

Each `p.show()` cell renders a **live WebGL widget** — rotate, zoom, and pan
directly in the notebook using your mouse/trackpad.

The VTK file is a **RectilinearGrid** covering the entire FFD domain
(plenum + white space). Cell data arrays:

| Array  | Description |
|--------|-------------|
| `T`    | Temperature [°C] |
| `U/V/W`| Velocity components [m/s] |
| `Speed`| Velocity magnitude [m/s] |
| `P`    | Pressure [Pa] |
| `Xi`   | Species fraction [-] |
| `FlagP`| Cell type (-1=fluid, 1=solid, 0=inlet, 2=outlet, …) |

In [1]:
import json
from pathlib import Path

import nest_asyncio2          # allows trame to autolaunch inside Jupyter's event loop
nest_asyncio2.apply()

import numpy as np
import pyvista as pv

# --------------------------------------------------------------------
# Backend: 'trame'  → interactive WebGL widget (rotate/zoom/pan in-cell)
#          'static' → inline PNG fallback (no trame required)
# --------------------------------------------------------------------
pv.set_jupyter_backend('trame')

# --------------------------------------------------------------------
# Path to the exported VTK file
# --------------------------------------------------------------------
REPO_ROOT = Path('..').resolve()
RUN_ID    = 'fnb-45s-v1'

RESULT_VTK = REPO_ROOT / 'output' / 'fnb' / 'run' / RUN_ID / 'results' / 'result.vtk'
GEO_JSON   = REPO_ROOT / 'ingest' / 'fnb' / 'current' / 'geometry.json'
PLOTS_DIR  = REPO_ROOT / 'output' / 'fnb' / 'run' / RUN_ID / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'VTK path : {RESULT_VTK}')
print(f'Exists   : {RESULT_VTK.exists()}')

VTK path : C:\Users\ZacharyCooper-Baldoc\Desktop\Repositories\CFD\output\fnb\run\fnb-45s-v1\results\result.vtk
Exists   : True


## 1 · Load and inspect the VTK grid

In [2]:
grid = pv.read(str(RESULT_VTK))

nx, ny, nz = grid.field_data['grid_nx_ny_nz']
print(f'Grid dimensions : {nx} x {ny} x {nz} = {grid.n_cells:,} cells')
print(f'Cell data arrays: {list(grid.cell_data.keys())}')
print(f'Bounds (m)      : X={grid.bounds[0]:.2f}–{grid.bounds[1]:.2f}  '
      f'Y={grid.bounds[2]:.2f}–{grid.bounds[3]:.2f}  '
      f'Z={grid.bounds[4]:.2f}–{grid.bounds[5]:.2f}')

if 'T' in grid.cell_data:
    T = grid.cell_data['T']
    valid = T[np.isfinite(T)]
    print(f'Temperature     : min={valid.min():.1f} °C  max={valid.max():.1f} °C  mean={valid.mean():.1f} °C')

if 'Speed' in grid.cell_data:
    spd = grid.cell_data['Speed']
    valid = spd[np.isfinite(spd)]
    print(f'Speed           : min={valid.min():.3f} m/s  max={valid.max():.3f} m/s')

Grid dimensions : 100 x 317 x 37 = 1,172,900 cells
Cell data arrays: ['U', 'V', 'W', 'T', 'P', 'Xi', 'FlagP', 'Speed']
Bounds (m)      : X=0.00–15.00  Y=0.00–25.00  Z=-0.60–3.90
Temperature     : min=14.0 °C  max=120.4 °C  mean=28.3 °C
Speed           : min=0.000 m/s  max=8.466 m/s


## 2 · Extract subsets by cell type

In [3]:
# Fluid cells: FlagP ≈ -1
fluid = grid.threshold(value=(-1.5, -0.5), scalars='FlagP')
print(f'Fluid cells : {fluid.n_cells:,}')

# Solid cells: FlagP ≈ 1  (racks + containment walls)
solids = grid.threshold(value=(0.5, 1.5), scalars='FlagP')
print(f'Solid cells : {solids.n_cells:,}')

# Plenum: Z < 0
plenum = fluid.clip(normal='z', origin=(0, 0, 0), invert=True)
print(f'Plenum cells: {plenum.n_cells:,}')

# White space only: Z >= 0
white_space = fluid.clip(normal='z', origin=(0, 0, 0), invert=False)
print(f'White-space : {white_space.n_cells:,}')

Fluid cells : 607,286
Solid cells : 565,320
Plenum cells: 0
White-space : 607,286


## 3 · Temperature — horizontal slices at rack inlet heights

**Interactive controls:** left-drag to rotate · scroll to zoom · right-drag to pan · double-click to reset view

In [4]:
INLET_HEIGHTS = [0.75, 1.13, 1.51]  # m above raised floor

p = pv.Plotter(shape=(1, 3))

for col, z_target in enumerate(INLET_HEIGHTS):
    p.subplot(0, col)
    slc = fluid.slice(normal='z', origin=(7.5, 12.5, z_target))
    p.add_mesh(
        slc, scalars='T', cmap='inferno',
        scalar_bar_args={'title': 'T [°C]', 'vertical': True},
    )
    p.add_mesh(solids, color='grey', opacity=0.15)
    p.view_xy()
    p.add_title(f'Temperature XY  z = {z_target} m', font_size=9)

p.link_views()
p.show()

Widget(value='<iframe src="http://localhost:53119/index.html?ui=P_0x1bd0a0cf6b0_0&reconnect=auto" class="pyvis…

## 4 · Temperature — vertical cross-sections

In [5]:
p = pv.Plotter(shape=(1, 2))

# YZ plane at room centreline X = 7.5 m
p.subplot(0, 0)
slc_yz = fluid.slice(normal='x', origin=(7.5, 12.5, 1.5))
p.add_mesh(slc_yz, scalars='T', cmap='inferno',
           scalar_bar_args={'title': 'T [°C]'})
p.add_mesh(solids, color='grey', opacity=0.15)
p.view_yz()
p.add_title('Temperature YZ  x = 7.5 m', font_size=9)

# XZ plane at room centreline Y = 12.5 m
p.subplot(0, 1)
slc_xz = fluid.slice(normal='y', origin=(7.5, 12.5, 1.5))
p.add_mesh(slc_xz, scalars='T', cmap='inferno',
           scalar_bar_args={'title': 'T [°C]'})
p.add_mesh(solids, color='grey', opacity=0.15)
p.view_xz()
p.add_title('Temperature XZ  y = 12.5 m', font_size=9)

p.link_views()
p.show()

Widget(value='<iframe src="http://localhost:53119/index.html?ui=P_0x1bd3aee5430_1&reconnect=auto" class="pyvis…

## 5 · Velocity speed — mid-height slice

In [6]:
p = pv.Plotter()

slc_spd = fluid.slice(normal='z', origin=(7.5, 12.5, 1.5))
p.add_mesh(
    slc_spd, scalars='Speed', cmap='viridis',
    scalar_bar_args={'title': 'Speed [m/s]'},
)
p.add_mesh(solids, color='#aaaaaa', opacity=0.20)
p.add_title('Velocity Speed — XY slice at z = 1.5 m', font_size=10)
p.view_xy()
p.show()

Widget(value='<iframe src="http://localhost:53119/index.html?ui=P_0x1bd3b74c170_2&reconnect=auto" class="pyvis…

## 6 · Velocity streamlines seeded from CRAC supply openings

In [7]:
if 'U' in fluid.cell_data and 'V' in fluid.cell_data and 'W' in fluid.cell_data:
    vel = np.column_stack([
        fluid.cell_data['U'],
        fluid.cell_data['V'],
        fluid.cell_data['W'],
    ])
    fluid_pts = fluid.cell_centers()
    fluid_pts['velocity'] = vel

    crac_supply_centres = np.array([
        [0.10,  4.45, -0.30],
        [14.90, 4.45, -0.30],
        [0.10, 12.25, -0.30],
        [14.90,12.25, -0.30],
        [0.10, 20.05, -0.30],
        [14.90,20.05, -0.30],
    ])
    seed_cloud = pv.PolyData(crac_supply_centres)

    streamlines = fluid_pts.streamlines_from_source(
        seed_cloud,
        vectors='velocity',
        max_time=30.0,
        initial_step_length=0.05,
        integration_direction='forward',
    )

    p = pv.Plotter()
    p.add_mesh(streamlines.tube(radius=0.03), scalars='Speed', cmap='plasma',
               scalar_bar_args={'title': 'Speed [m/s]'})
    p.add_mesh(solids, color='#cccccc', opacity=0.18)
    p.add_title('Velocity Streamlines from CRAC Supplies', font_size=10)
    p.show()
else:
    print('Velocity arrays (U, V, W) not found in VTK — skipping streamlines.')

C:\Users\ZacharyCooper-Baldoc\AppData\Local\Temp\ipykernel_58956\2989362812.py:20: PyVistaDeprecationWarning: ``max_time`` parameter is deprecated.  It will be removed in v0.48
  streamlines = fluid_pts.streamlines_from_source(


KeyError: 'Data array (Speed) not present in this dataset.'

## 7 · Combined 3-D overview: temperature volume + solid geometry

Full 3-D interactive view — use the widget controls to orbit around the data centre.

In [8]:
p = pv.Plotter()

# Temperature volume: upper half of white space
clip = fluid.clip(normal='z', origin=(7.5, 12.5, 1.5), invert=False)
p.add_mesh(
    clip, scalars='T', cmap='inferno', opacity=0.6,
    scalar_bar_args={'title': 'T [°C]', 'vertical': True},
)

# Solid racks + containment walls
p.add_mesh(solids, color='#888888', opacity=0.45)

# CRAC supply markers
crac_pts = pv.PolyData(np.array([
    [0.10,  4.45, -0.30], [14.90, 4.45, -0.30],
    [0.10, 12.25, -0.30], [14.90,12.25, -0.30],
    [0.10, 20.05, -0.30], [14.90,20.05, -0.30],
]))
p.add_mesh(crac_pts, color='cyan', point_size=12, render_points_as_spheres=True,
           label='CRAC supply')

# Return grille markers
ret_pts = pv.PolyData(np.array([
    [0.15,  4.45, 3.0], [14.85, 4.45, 3.0],
    [0.15, 12.25, 3.0], [14.85,12.25, 3.0],
    [0.15, 20.05, 3.0], [14.85,20.05, 3.0],
]))
p.add_mesh(ret_pts, color='blue', point_size=12, render_points_as_spheres=True,
           label='CRAC return')

p.add_legend()
p.add_title('FNB Data Centre — Temperature + Geometry', font_size=10)
p.show()

Widget(value='<iframe src="http://localhost:53119/index.html?ui=P_0x1bd3b74fda0_4&reconnect=auto" class="pyvis…

## 8 · Hot aisle containment detail — ha_12

In [9]:
ha_region = fluid.clip_box(bounds=(0, 15, 3.5, 5.5, -0.6, 3.9), invert=False)
ha_solids = solids.clip_box(bounds=(0, 15, 3.5, 5.5, -0.6, 3.9), invert=False)

p = pv.Plotter()
p.add_mesh(ha_region, scalars='T', cmap='inferno', opacity=0.7,
           scalar_bar_args={'title': 'T [°C]'})
p.add_mesh(ha_solids, color='#777777', opacity=0.5)
p.add_title('Hot Aisle ha_12 — Temperature Detail', font_size=9)
p.show()

Widget(value='<iframe src="http://localhost:53119/index.html?ui=P_0x1bd3b74dee0_5&reconnect=auto" class="pyvis…

## 9 · Sensor-location temperature lookup

In [ ]:
import csv as _csv

sensor_csv = REPO_ROOT / 'ingest' / 'fnb' / 'current' / 'sensor_locations.csv'

if sensor_csv.exists():
    sensors: list[dict] = []
    with sensor_csv.open(newline='', encoding='utf-8') as fh:
        for row in _csv.DictReader(fh):
            sensors.append(row)

    pts = np.array([[float(s['x']), float(s['y']), float(s['z'])] for s in sensors])
    sensor_cloud = pv.PolyData(pts)
    sampled = grid.sample(sensor_cloud)
    T_at_sensors = sampled['T']

    print(f"{'Sensor':<25} {'x':>6} {'y':>7} {'z':>6} {'T [°C]':>8}")
    print('-' * 60)
    for i, s in enumerate(sensors[:20]):
        t_val = T_at_sensors[i]
        t_str = f'{t_val:.1f}' if np.isfinite(t_val) else 'N/A'
        print(f"{s['sensor_id']:<25} {float(s['x']):>6.2f} {float(s['y']):>7.2f} "
              f"{float(s['z']):>6.2f} {t_str:>8}")
    if len(sensors) > 20:
        print(f'  … {len(sensors) - 20} more sensors')
else:
    print('sensor_locations.csv not found — run generate-ingest first.')